# Frobenius Motif Similarity Analysis

Visually cluster the cropped motif images in `frobenius_artifacts/analysis/motifs/` by appearance.

**Pipeline:**
1. Scan motif crops and load metadata from filenames
2. Embed every crop with **CLIP ViT-B/32** (512-dim, cosine space)
3. Cache embeddings so subsequent runs are instant
4. Project to 2-D with **t-SNE** for layout
5. Cluster with **HDBSCAN** for groupings
6. **Scatter map** — every motif as a thumbnail, neighbours are visually similar
7. **Cluster gallery** — grids of same-cluster motifs
8. **Cosine similarity heatmap**
9. **Nearest-neighbour explorer** — pick any motif, see its closest matches

```bash
uv run --project src/python jupyter notebook src/python/motif_similarity.ipynb
```

> Run `extract_crops.py` first if the motifs directory is empty.

In [ ]:
# ── Cell 1: Colab setup (skip locally) ────────────────────────────────────
import sys
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "open-clip-torch", "hdbscan", "scikit-learn",
        "Pillow", "numpy>=1.24,<2", "ipywidgets",
    ], check=True)
    print("Colab: deps installed.")
else:
    print("Local — skipping Colab setup.")

In [ ]:
# ── Cell 2: imports & paths ────────────────────────────────────────────────
import sys, os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

import torch
import open_clip

from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import hdbscan

# ── Paths ──────────────────────────────────────────────────────────────────
ON_COLAB = "google.colab" in sys.modules

if ON_COLAB:
    MOTIFS_DIR = Path("/content/motifs")
    CACHE_DIR  = Path("/content")
else:
    REPO_ROOT  = Path("../..")
    MOTIFS_DIR = REPO_ROOT / "frobenius_artifacts/analysis/motifs"
    CACHE_DIR  = Path(".")   # src/python/ — keeps cache next to notebook

EMBED_CACHE = CACHE_DIR / "motif_embeddings.npy"
PATHS_CACHE = CACHE_DIR / "motif_paths.txt"

print(f"Motifs dir : {MOTIFS_DIR.resolve()}")
print(f"Cache dir  : {CACHE_DIR.resolve()}")
if not MOTIFS_DIR.exists():
    print("WARNING: motifs directory not found — run extract_crops.py first.")

In [ ]:
# ── Cell 3: scan motif crops & build metadata ──────────────────────────────
# Expected layout:
#   motifs/<panel_stem>/<NNN>_<scale>_iou<X.XXX>.png

records = []
for panel_dir in sorted(MOTIFS_DIR.iterdir()):
    if not panel_dir.is_dir():
        continue
    panel_name = panel_dir.name
    for img_path in sorted(panel_dir.glob("*.png")):
        stem   = img_path.stem          # e.g. "003_motif_iou0.889"
        parts  = stem.split("_")
        idx    = int(parts[0])
        scale  = parts[1]               # motif | register
        iou    = float(parts[2].replace("iou", ""))
        records.append({
            "path"    : img_path,
            "panel"   : panel_name,
            "index"   : idx,
            "scale"   : scale,
            "pred_iou": iou,
        })

panels = sorted(set(r["panel"] for r in records))
print(f"{len(records)} crops across {len(panels)} panels")
for p in panels:
    n = sum(1 for r in records if r["panel"] == p)
    print(f"  {p}: {n}")

In [ ]:
# ── Cell 4: CLIP embeddings (cached) ──────────────────────────────────────
# First run downloads ~340 MB CLIP ViT-B/32 checkpoint.
# Subsequent runs load the cached .npy in <1 s.

CLIP_MODEL   = "ViT-B-32"
CLIP_WEIGHTS = "openai"
BATCH        = 32

def _resolve_device():
    if torch.cuda.is_available(): return "cuda"
    return "cpu"   # MPS excluded: open-clip has float32/float16 issues on MPS

_clip_cache = {}

def get_clip():
    if not _clip_cache:
        device = _resolve_device()
        print(f"Loading CLIP {CLIP_MODEL} ({CLIP_WEIGHTS}) on {device}...")
        model, _, preprocess = open_clip.create_model_and_transforms(
            CLIP_MODEL, pretrained=CLIP_WEIGHTS
        )
        model = model.to(device).eval()
        _clip_cache.update({"model": model, "prep": preprocess, "device": device})
        print("CLIP loaded.")
    return _clip_cache["model"], _clip_cache["prep"], _clip_cache["device"]


def compute_embeddings(recs):
    model, prep, device = get_clip()
    all_feats = []
    for i in range(0, len(recs), BATCH):
        batch = recs[i : i + BATCH]
        imgs  = torch.stack([prep(Image.open(r["path"]).convert("RGB")) for r in batch]).to(device)
        with torch.no_grad():
            feats = model.encode_image(imgs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        all_feats.append(feats.cpu().float().numpy())
        print(f"  {min(i + BATCH, len(recs))}/{len(recs)}", end="\r")
    print()
    return np.vstack(all_feats)


current_paths = [str(r["path"]) for r in records]

if (
    EMBED_CACHE.exists() and PATHS_CACHE.exists()
    and PATHS_CACHE.read_text().strip().split("\n") == current_paths
):
    print("Loading cached embeddings...")
    embeddings = np.load(EMBED_CACHE)
    print(f"  shape: {embeddings.shape}")
else:
    print("Computing CLIP embeddings (first run downloads ~340 MB)...")
    embeddings = compute_embeddings(records)
    np.save(EMBED_CACHE, embeddings)
    PATHS_CACHE.write_text("\n".join(current_paths))
    print(f"Computed and cached: {embeddings.shape}")

In [ ]:
# ── Cell 5: t-SNE projection + HDBSCAN clustering + similarity matrix ──────

N = len(records)

# t-SNE — cosine metric via pre-normalised embeddings (L2 distance ≈ cosine distance)
PERPLEXITY = min(30, max(5, N // 8))
print(f"Running t-SNE (n={N}, perplexity={PERPLEXITY})...")
tsne = TSNE(
    n_components=2,
    perplexity=PERPLEXITY,
    metric="euclidean",   # embeddings already L2-normalised so euclidean ≈ cosine
    random_state=42,
    n_iter=1500,
    init="pca",
)
coords = tsne.fit_transform(embeddings)
print(f"t-SNE done. Layout range x={coords[:,0].min():.1f}–{coords[:,0].max():.1f}, "
      f"y={coords[:,1].min():.1f}–{coords[:,1].max():.1f}")

# HDBSCAN on the L2-normalised embeddings
MIN_CLUSTER = max(3, N // 40)   # at least ~2.5% of images per cluster
print(f"\nHDBSCAN (min_cluster_size={MIN_CLUSTER})...")
clusterer  = hdbscan.HDBSCAN(
    min_cluster_size=MIN_CLUSTER,
    min_samples=2,
    metric="euclidean",
)
labels = clusterer.fit_predict(embeddings)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = int((labels == -1).sum())
print(f"{n_clusters} clusters, {n_noise}/{N} unclustered (noise)")
for c in sorted(set(labels)):
    tag = "noise" if c == -1 else f"cluster {c:2d}"
    print(f"  {tag}: {(labels == c).sum()}")

# Cosine similarity matrix (sorted by cluster for the heatmap)
sim_matrix = cosine_similarity(embeddings)

In [ ]:
# ── Cell 6: similarity map — thumbnails on t-SNE axes ─────────────────────
# Every motif is drawn as a small image at its t-SNE position.
# Colour of the border = cluster assignment.
# Visually similar motifs cluster together.

THUMB = 40          # thumbnail size (pixels)
ZOOM  = 0.50        # OffsetImage zoom factor

# Build colour palette
_PALETTE = list(plt.cm.tab20.colors) + list(plt.cm.tab20b.colors)
_NOISE_COL = (0.55, 0.55, 0.55)

def cluster_color(lbl):
    return _NOISE_COL if lbl == -1 else _PALETTE[lbl % len(_PALETTE)]

# Load + resize thumbnails
print("Loading thumbnails...")
thumbs = []
for r in records:
    img = Image.open(r["path"]).convert("RGB")
    img.thumbnail((THUMB, THUMB))
    # Pad to square so OffsetImage doesn't distort layout
    sq = Image.new("RGB", (THUMB, THUMB), (30, 30, 30))
    sq.paste(img, ((THUMB - img.width) // 2, (THUMB - img.height) // 2))
    thumbs.append(np.array(sq))
print(f"  {len(thumbs)} thumbnails ready.")

# Draw scatter
fig, ax = plt.subplots(figsize=(20, 16))
ax.set_facecolor("#111")
fig.patch.set_facecolor("#111")

# Border patches first (drawn as squares behind the images)
BORDER = 2
half = ZOOM * THUMB / 2
for i, (thumb, (x, y)) in enumerate(zip(thumbs, coords)):
    col = cluster_color(labels[i])
    ax.add_patch(plt.Rectangle(
        (x - half - BORDER, y - half - BORDER),
        2 * half + 2 * BORDER, 2 * half + 2 * BORDER,
        color=col, zorder=1, linewidth=0,
    ))

# Images on top
for i, (thumb, (x, y)) in enumerate(zip(thumbs, coords)):
    ab = AnnotationBbox(OffsetImage(thumb, zoom=ZOOM), (x, y),
                        frameon=False, zorder=2)
    ax.add_artist(ab)

ax.autoscale_view()
ax.set_xlim(coords[:,0].min() - THUMB, coords[:,0].max() + THUMB)
ax.set_ylim(coords[:,1].min() - THUMB, coords[:,1].max() + THUMB)
ax.axis("off")
ax.set_title(
    f"Motif similarity map  |  {N} crops  |  t-SNE (CLIP ViT-B/32)  |  "
    f"{n_clusters} HDBSCAN clusters + {n_noise} unclustered",
    color="white", fontsize=12, pad=10,
)

# Legend
legend_handles = [mpatches.Patch(color=_NOISE_COL, label=f"unclustered ({n_noise})")]
for c in sorted(set(labels)):
    if c == -1: continue
    cnt = int((labels == c).sum())
    legend_handles.append(mpatches.Patch(color=cluster_color(c), label=f"C{c} (n={cnt})"))
ax.legend(handles=legend_handles, loc="lower right", fontsize=8,
          facecolor="#222", labelcolor="white", framealpha=0.85,
          ncols=max(1, len(legend_handles) // 12))

plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 7: cosine similarity heatmap ─────────────────────────────────────
# Images sorted by cluster. Bright = high similarity.
# Block structure along the diagonal reveals the clusters.

order = np.argsort(labels)   # sort by cluster label (-1 noise goes first)
sim_sorted = sim_matrix[np.ix_(order, order)]

# Cluster boundary tick positions
sorted_labels = labels[order]
boundaries = [0]
for i in range(1, N):
    if sorted_labels[i] != sorted_labels[i - 1]:
        boundaries.append(i)
boundaries.append(N)

fig_size = min(18, max(8, N / 14))
fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))
im = ax.imshow(sim_sorted, cmap="inferno", vmin=0.5, vmax=1.0, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02, label="cosine similarity")

# Draw cluster boundary lines
for b in boundaries[1:-1]:
    ax.axhline(b - 0.5, color="cyan", lw=0.6, alpha=0.6)
    ax.axvline(b - 0.5, color="cyan", lw=0.6, alpha=0.6)

# Cluster labels on axes
tick_pos   = [(boundaries[i] + boundaries[i + 1]) / 2 for i in range(len(boundaries) - 1)]
tick_lbls  = []
for c in [sorted_labels[int(p)] for p in tick_pos]:
    tick_lbls.append("noise" if c == -1 else f"C{c}")

ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lbls, rotation=60, fontsize=7)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_lbls, fontsize=7)
ax.set_title("Pairwise cosine similarity (sorted by cluster)", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Cell 8: cluster gallery ────────────────────────────────────────────────
# For each cluster: a row of thumbnails, sized proportionally to cluster size.
# Unclustered motifs are shown last.

GALLERY_THUMB = 80   # px per image in the gallery
MAX_COLS      = 20   # max images per row before wrapping

cluster_ids = sorted(c for c in set(labels) if c != -1) + ([-1] if -1 in labels else [])

for c in cluster_ids:
    idx_in_cluster = [i for i, lbl in enumerate(labels) if lbl == c]
    # Sort within cluster by similarity to cluster centroid (best members first)
    centroid = embeddings[idx_in_cluster].mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    sims_to_centroid = embeddings[idx_in_cluster] @ centroid
    idx_in_cluster = [idx_in_cluster[j] for j in np.argsort(-sims_to_centroid)]

    tag   = f"Cluster {c}" if c != -1 else "Unclustered (noise)"
    color = cluster_color(c)
    n     = len(idx_in_cluster)

    n_cols = min(n, MAX_COLS)
    n_rows = (n + n_cols - 1) // n_cols
    fig_w  = n_cols * (GALLERY_THUMB / 72)
    fig_h  = n_rows * (GALLERY_THUMB / 72) + 0.4

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h),
                             squeeze=False)
    fig.patch.set_facecolor("#1a1a1a")
    fig.suptitle(f"{tag}  (n={n})", color="white", fontsize=9,
                 x=0.02, ha="left", y=1.0)

    for ax_row in axes:
        for ax in ax_row:
            ax.axis("off")
            ax.set_facecolor("#1a1a1a")

    for j, gi in enumerate(idx_in_cluster):
        r_ax = axes[j // n_cols][j % n_cols]
        img  = Image.open(records[gi]["path"]).convert("RGB")
        r_ax.imshow(img)
        r_ax.set_title(
            f"{records[gi]['index']}\n{records[gi]['panel'][:14]}",
            fontsize=4, color="white", pad=1,
        )
        for spine in r_ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(1.5)
            spine.set_visible(True)

    plt.tight_layout(pad=0.2)
    plt.show()

In [ ]:
# ── Cell 9: nearest-neighbour explorer ────────────────────────────────────
# Pick any motif from the dropdown; the widget shows its top-K visual matches.

TOP_K = 12   # how many nearest neighbours to display

# Dropdown options: "panel/NNN_scale (iou=X.XX)"
def _label(r):
    return f"{r['panel'][:28]} / {r['index']:03d} {r['scale']} (iou={r['pred_iou']:.2f})"

picker = widgets.Dropdown(
    options=[(  _label(r), i) for i, r in enumerate(records)],
    value=0,
    description="Query:",
    layout=widgets.Layout(width="80%"),
    style={"description_width": "55px"},
)
w_k = widgets.IntSlider(
    min=4, max=40, step=2, value=TOP_K,
    description="Top K:",
    continuous_update=False,
    style={"description_width": "55px"},
    layout=widgets.Layout(width="40%"),
)
w_filter_cluster = widgets.Checkbox(
    value=False,
    description="Same cluster only",
    style={"description_width": "initial"},
)
out = widgets.Output()

def _show_nn(query_idx, k, same_cluster_only):
    out.clear_output(wait=True)
    sims = sim_matrix[query_idx].copy()
    sims[query_idx] = -1   # exclude self

    if same_cluster_only and labels[query_idx] != -1:
        mask = labels != labels[query_idx]
        sims[mask] = -1

    top_k_idx = np.argsort(-sims)[:k]
    top_k_sim = sims[top_k_idx]

    n_cols = min(k, 8)
    n_rows = 1 + (k + n_cols - 1) // n_cols   # +1 row for query
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.4, n_rows * 1.6),
                             squeeze=False)
    fig.patch.set_facecolor("#1a1a1a")
    for row in axes:
        for ax in row:
            ax.axis("off")
            ax.set_facecolor("#1a1a1a")

    # Row 0: query image (spans all columns via colspan trick — just use col 0)
    qr = records[query_idx]
    axes[0][0].imshow(Image.open(qr["path"]).convert("RGB"))
    axes[0][0].set_title(
        f"QUERY\n#{qr['index']} {qr['scale']}\n{qr['panel'][:22]}",
        fontsize=6, color="white", pad=2,
    )
    for spine in axes[0][0].spines.values():
        spine.set_edgecolor("gold"); spine.set_linewidth(2); spine.set_visible(True)

    # Annotate cluster
    qlbl = labels[query_idx]
    axes[0][0].text(
        0.5, -0.05,
        f"C{qlbl}" if qlbl != -1 else "noise",
        transform=axes[0][0].transAxes,
        ha="center", fontsize=6,
        color=cluster_color(qlbl),
    )

    # Remaining result row 0 slots: blank
    for col in range(1, n_cols):
        axes[0][col].set_visible(False)

    # Nearest neighbours
    for j, (ni, sim_val) in enumerate(zip(top_k_idx, top_k_sim)):
        row = 1 + j // n_cols
        col = j % n_cols
        nr  = records[ni]
        axes[row][col].imshow(Image.open(nr["path"]).convert("RGB"))
        axes[row][col].set_title(
            f"sim={sim_val:.3f}\n#{nr['index']} {nr['scale']}\n{nr['panel'][:22]}",
            fontsize=5, color="white", pad=2,
        )
        nlbl = labels[ni]
        col_nn = cluster_color(nlbl)
        for spine in axes[row][col].spines.values():
            spine.set_edgecolor(col_nn); spine.set_linewidth(1.5); spine.set_visible(True)

    plt.suptitle(
        f"Top-{k} nearest neighbours for motif {qr['index']} ({qr['panel'][:30]})",
        color="white", fontsize=9, y=1.01,
    )
    plt.tight_layout(pad=0.4)
    with out:
        plt.show()


def _on_change(_):
    _show_nn(picker.value, w_k.value, w_filter_cluster.value)

picker.observe(_on_change, names="value")
w_k.observe(_on_change, names="value")
w_filter_cluster.observe(_on_change, names="value")

display(
    widgets.HTML("<h3 style='margin-bottom:4px'>Nearest-neighbour explorer</h3>"),
    picker,
    widgets.HBox([w_k, w_filter_cluster]),
    out,
)
_show_nn(picker.value, w_k.value, w_filter_cluster.value)